In [ ]:
%run "./00_CC_Mapping_Setup.ipynb" 

In [ ]:
%sql
/* ===================================================================================
   METRIC: TP02 - TP High Value
   
   DATA SOURCES:
   1. Master AU List: hive_metastore.ra_adido_2025.fy25_list_of_units
   2. TP Unit Mapping: hive_metastore.ra_adido_2025.third_party_unit_mapping
   3. Third Party Data: hive_metastore.ra_adido_2025.abac_request_paul_v3
   
   BUSINESS QUESTION: 
   Percentage of "high value" business arrangements/contracts maintained by the unit 
   during the assessment period.
   
   LOGIC & ARCHITECTURE SUMMARY:
   1. MASTER AU ANCHOR: Uses the Master AU list as the absolute key for the output. 
      Only AUs in this filtered list (Canada, HK, Barbados + US_OR_CANADA = 'CANADA') 
      will appear in the final report.
   2. HIGHEST VALUE FILTER: Cleans the 'Contract_Amount' string (removes commas/$) 
      and casts to a Double. The Numerator evaluates contracts >= 1,000,000.
   3. FLATTEN LOBs: Columns K (owning_lob) and L (lob_using) contain comma-separated 
      lists of business units. Uses LATERAL VIEW explode(split(...)) to expand these 
      strings into individual evaluation rows so no unit is missed.
   4. SAFE MAPPING: Maps expanded LOBs to TPRM_Units using standard LIKE syntax 
      ('%' || String || '%') to avoid regex failures on special characters ('&', '-').
   5. NAME BRIDGE: The mapping table stores the AU Name in the Assessable_Unit_ID 
      column. This logic bridges that String Name to the Master List's AU_Name to 
      retrieve the true Numeric BusinessID.
   6. AGGREGATING (NUMERATOR & DENOMINATOR): Counts DISTINCT EngagementNumbers per AU 
      (Denominator) and DISTINCT EngagementNumbers >= 1MM (Numerator) to remove 
      duplicates caused by the flattening process.
   7. FINAL OUTPUT: Calculates the percentage (Numerator/Denominator * 100), rounds 
      to 2 decimal places, appends a '%' sign, and outputs the strict 6 columns.
=================================================================================== */

WITH Master_AUs AS (
    -- 1. Grab Master List data: This is our strict anchor for the final output
    SELECT 
        TRIM(CAST(BusinessID AS STRING)) AS BusinessID,
        TRIM(Business_Operational_Unit_Name) AS AU_Name,
        TRIM(BusinessSegment) AS Business_Segment,
        'Yes' AS In_ABAC_AU_List
    FROM hive_metastore.ra_adido_2025.fy25_list_of_units
    WHERE BusinessID IS NOT NULL
      AND UPPER(TRIM(Jurisdiction)) IN ('CANADA', 'HONG KONG', 'BARBADOS')
      AND UPPER(TRIM(US_OR_CANADA)) = 'CANADA'
),

Mapped_AUs AS (
    -- 2. Grab valid TPRM Mapping Strings and block blank/null strings
    -- Note: Assessable_Unit_ID actually contains the string Name, not the numeric ID
    SELECT DISTINCT 
        TRIM(Assessable_Unit_ID) AS Mapping_AU_Name,
        TRIM(TPRM_Units) AS TPRM_Units
    FROM hive_metastore.ra_adido_2025.third_party_unit_mapping
    WHERE Assessable_Unit_ID IS NOT NULL
      AND NULLIF(TRIM(TPRM_Units), '') IS NOT NULL
),

Third_Party_Risk_Data AS (
    -- 3. Extract TP data, clean the contract amount string, and cast to double
    SELECT 
        EngagementNumber,
        owning_lob,
        lob_using,
        TRY_CAST(REPLACE(REPLACE(TRIM(Contract_Amount), ',', ''), '$', '') AS DOUBLE) AS Cleaned_Amount
    FROM hive_metastore.ra_adido_2025.abac_request_paul_v3
    WHERE EngagementNumber IS NOT NULL
),

Flattened_LOBs AS (
    -- 4. FLATTEN RULE: Merge Col K and L, split by commas, and expand into individual rows
    SELECT 
        EngagementNumber, 
        Cleaned_Amount,
        TRIM(exploded_lob) AS Expanded_LOB
    FROM Third_Party_Risk_Data
    LATERAL VIEW explode(split(concat_ws(',', owning_lob, lob_using), ',')) AS exploded_lob
    WHERE exploded_lob IS NOT NULL AND TRIM(exploded_lob) != ''
),

Engagements_By_AU AS (
    -- 5. Bridge mappings to Master List and calculate Numerator/Denominator
    SELECT 
        mast.BusinessID,
        -- Total Distinct Engagements mapped to this AU
        COUNT(DISTINCT f.EngagementNumber) AS Denominator,
        -- Distinct Engagements mapped to this AU that are >= 1 Million
        COUNT(DISTINCT CASE WHEN f.Cleaned_Amount >= 1000000 THEN f.EngagementNumber END) AS Numerator
    FROM Flattened_LOBs f
    -- Safe Mapping Join
    INNER JOIN Mapped_AUs map
        ON UPPER(f.Expanded_LOB) LIKE '%' || UPPER(map.TPRM_Units) || '%'
    -- Name Bridge Join
    INNER JOIN Master_AUs mast
        ON UPPER(TRIM(map.Mapping_AU_Name)) = UPPER(TRIM(mast.AU_Name))
    GROUP BY mast.BusinessID
)

-- 6. Final Output: Strict 6-column structure anchored to Master AU List
SELECT 
    a.BusinessID,                          
    a.AU_Name,                             
    a.Business_Segment,
    'TP02' AS QuestionID,               
    
    -- Math: Handles division by zero safely, calculates percentage, and appends '%'
    CASE 
        WHEN COALESCE(e.Denominator, 0) = 0 THEN '0%'
        ELSE CAST(ROUND((CAST(e.Numerator AS DOUBLE) / e.Denominator) * 100, 2) AS STRING) || '%'
    END AS Response,
    
    a.In_ABAC_AU_List
    
FROM Master_AUs a
LEFT JOIN Engagements_By_AU e 
    ON a.BusinessID = e.BusinessID
ORDER BY a.BusinessID;

In [ ]:
%sql
/* ===================================================================================
   DEBUG TABLE: TP02 - High Value Contract Traceability
   
   PURPOSE: Shows the raw Contract Amount vs Cleaned Amount, prints the full 
   comma-separated LOB strings vs the exploded chunks, and displays the Master AU 
   lookup status to troubleshoot mapping drops.
=================================================================================== */

WITH Master_AUs AS (
    SELECT 
        TRIM(CAST(BusinessID AS STRING)) AS Master_Numeric_ID,
        TRIM(Business_Operational_Unit_Name) AS Master_AU_Name
    FROM hive_metastore.ra_adido_2025.fy25_list_of_units
),

Flattened AS (
    -- Extract raw data, clean the amount, and explode the LOBs into individual rows
    SELECT 
        p.EngagementNumber,
        p.Contract_Amount AS Raw_Contract_Amount,
        TRY_CAST(REPLACE(REPLACE(TRIM(p.Contract_Amount), ',', ''), '$', '') AS DOUBLE) AS Cleaned_Amount,
        p.owning_lob AS Full_Col_K,
        p.lob_using AS Full_Col_L,
        TRIM(exploded_lob) AS Exploded_Chunk
    FROM hive_metastore.ra_adido_2025.abac_request_paul_v3 p
    LATERAL VIEW explode(split(concat_ws(',', owning_lob, lob_using), ',')) AS exploded_lob
    WHERE p.EngagementNumber IS NOT NULL
      AND exploded_lob IS NOT NULL AND TRIM(exploded_lob) != ''
)

SELECT DISTINCT
    f.EngagementNumber,
    
    -- Contract Value Traceability
    f.Raw_Contract_Amount,
    f.Cleaned_Amount,
    CASE WHEN f.Cleaned_Amount >= 1000000 THEN 'Yes - Counts in Numerator' ELSE 'No - Denominator Only' END AS Is_High_Value,
    
    -- LOB Flattening Traceability
    f.Full_Col_K,
    f.Full_Col_L,
    f.Exploded_Chunk,
    
    -- Mapping & Bridging Traceability
    map.TPRM_Units AS Matched_Mapping_String,
    map.Assessable_Unit_ID AS Mapping_Table_AU_Name,
    mast.Master_Numeric_ID AS Bridged_Master_ID,
    CASE WHEN mast.Master_Numeric_ID IS NULL THEN 'MISSING FROM MASTER LIST' ELSE 'SUCCESSFULLY BRIDGED' END AS Master_List_Status
    
FROM Flattened f
-- Safe Mapping Join
LEFT JOIN hive_metastore.ra_adido_2025.third_party_unit_mapping map 
    ON UPPER(f.Exploded_Chunk) LIKE '%' || UPPER(TRIM(map.TPRM_Units)) || '%'
-- Name Bridge Join
LEFT JOIN Master_AUs mast 
    ON UPPER(TRIM(map.Assessable_Unit_ID)) = UPPER(TRIM(mast.Master_AU_Name))
ORDER BY f.EngagementNumber;